In [149]:
from dataclasses import dataclass
import numpy as np

@dataclass
class Grid:
    rows: int
    cols: int
    step_reward: int
    terminals: dict
    walls: set
    noise: float = 0.0
    actions = dict(up=(-1, 0), right=(0, 1), down=(1, 0), left=(0, -1))
    arrows = {'up': "↑", 'right': "→", 'down': "↓", 'left': "←"}
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return self.terminals[cell]
        elif cell in self.walls:
            return '#'
        else:
            return '·'
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def step(self, cell, action):
        result = []
        action_keys = list(self.actions.keys())
        aind = list(self.actions.keys()).index(action)
        n_actions = len(action_keys)
        for a, prob in zip(
            [aind, (aind + 1) % n_actions, (aind - 1) % n_actions],
            [1 - self.noise, self.noise / 2, self.noise / 2],
        ):
            movement = self.actions[action_keys[a]]
            next_cell = (cell[0] + movement[0], cell[1] + movement[1])
            outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
            on_wall = next_cell in self.walls
            if outside or on_wall:
                next_cell = cell

            if next_cell in self.terminals:
                reward = self.terminals[next_cell]
            else:
                reward = self.step_reward
            result.append((prob, next_cell, reward))
        return result
    def properties(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
    noise={self.noise},
)
'''


default_grid = Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
    noise=0.2,
)
default_grid

Grid(rows=3, cols=4, step_reward=0, terminals={(0, 3): 1}, walls={(1, 1)}, noise=0.2)

In [150]:
import io, contextlib, functools

def silent(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        with contextlib.redirect_stdout(io.StringIO()):
            return fn(*args, **kwargs)
    return wrapper

In [151]:
def render_policy(grid: Grid, policy):
    for r in range(len(policy)):
        for c in range(len(policy[0])):
            if r == 0 and c == 0:
                print("r/c", end="")
                print(''.join([f'{v:>2} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>2} ', end="")

            if policy[r][c] != None:
                value = grid.arrows[max(policy[r][c], key=policy[r][c].get)]
            else:
                value = grid.cell_repr(r, c)
            print(f' {value} ', end='')
        print()
    print('------------------')

def show_V(grid: Grid, V):
    for r in range(grid.rows):
        for c in range(grid.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")
            print(f'{round(V[r][c], 2):>4} ', end="")
        print()
    print('------------------')


def value_iteration(grid: Grid, gamma=0.9, theta=1e-6, max_iters=1000):
    print('---------- value_iteration ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                def q(action):
                    result = grid.step(cell, action)
                    q_value = 0
                    for prob, next_cell, reward in result:
                        q_value += prob * (
                            reward + gamma * V_old[next_cell[0]][next_cell[1]]
                        )
                    return q_value
                new_value = float('-inf')
                for action in grid.actions:
                    value = q(action)
                    if value > new_value:
                        new_value = value
                        policy[r][c] = {action: 1.0}
                V[r][c] = new_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        show_V(grid, V)
        i += 1
    render_policy(grid, policy)
    converged = delta <= theta
    if converged:
        print(f'value_iteration converged in {i} iterations')
    else:
        print(f'value_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [152]:
policy, V, converged = value_iteration(default_grid);

---------- value_iteration ------------
  r/c   0    1    2    3 
   0  0.0  0.0  0.8    0 
   1  0.0    0  0.0  0.8 
   2  0.0  0.0  0.0  0.0 
------------------
  r/c   0    1    2    3 
   0  0.0 0.58 0.87    0 
   1  0.0    0 0.65 0.87 
   2  0.0  0.0  0.0 0.58 
------------------
  r/c   0    1    2    3 
   0 0.41 0.73 0.94    0 
   1  0.0    0 0.76 0.94 
   2  0.0  0.0 0.52 0.68 
------------------
  r/c   0    1    2    3 
   0 0.56 0.81 0.95    0 
   1  0.3    0 0.83 0.95 
   2  0.0 0.37 0.61 0.78 
------------------
  r/c   0    1    2    3 
   0 0.66 0.83 0.96    0 
   1 0.46    0 0.85 0.96 
   2  0.3 0.51  0.7 0.81 
------------------
  r/c   0    1    2    3 
   0  0.7 0.84 0.96    0 
   1 0.56    0 0.85 0.96 
   2 0.43  0.6 0.73 0.83 
------------------
  r/c   0    1    2    3 
   0 0.72 0.84 0.96    0 
   1  0.6    0 0.86 0.96 
   2 0.52 0.63 0.74 0.83 
------------------
  r/c   0    1    2    3 
   0 0.73 0.85 0.96    0 
   1 0.63    0 0.86 0.96 
   2 0.56 0.65 0.75 0

In [153]:
def read_policy(grid: Grid, V, gamma=0.9, incumbent_policy=None):
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            cell = (r, c)
            if cell in grid.walls or cell in grid.terminals:
                continue
            def q(action):
                result = grid.step(cell, action)
                q_value = 0
                for prob, next_cell, reward in result:
                    q_value += prob * (
                        reward + gamma * V[next_cell[0]][next_cell[1]]
                    )
                return q_value
            new_action = max(grid.actions, key=q)
            if incumbent_policy:
                incumbent_action = max(incumbent_policy[r][c], key=incumbent_policy[r][c].get)
                if q(new_action) - q(incumbent_action) < 1e-9:
                    new_action = incumbent_action
            policy[r][c] = {new_action: 1.0}
    return policy


policy = read_policy(default_grid, V)
policy

[[{'right': 1.0}, {'right': 1.0}, {'right': 1.0}, None],
 [{'up': 1.0}, None, {'up': 1.0}, {'up': 1.0}],
 [{'right': 1.0}, {'right': 1.0}, {'up': 1.0}, {'up': 1.0}]]

In [154]:
render_policy(default_grid, policy)

r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------


In [155]:
def policy_evaluation(
    grid: Grid, policy, gamma=0.9, theta=1e-6, verbose=False, max_iters=1000
):
    if verbose:
        print('---------- policy_evaluation ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                value = 0
                def q(action):
                    result = grid.step(cell, action)
                    q_value = 0
                    for prob, next_cell, reward in result:
                        q_value += prob * (
                            reward + gamma * V_old[next_cell[0]][next_cell[1]]
                        )
                    return q_value
                for action, prob in policy[r][c].items():
                    value += prob * q(action)
                V[r][c] = value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            show_V(grid, V)
        i += 1
    converged = delta <= theta
    if converged:
        print(f'policy_evaluation converged in {i} iterations')
    else:
        print(f'policy_evaluation did not converge in {max_iters} iterations')
    return V, converged


policy_evaluation(default_grid, policy, verbose=True);

---------- policy_evaluation ------------
  r/c   0    1    2    3 
   0  0.0  0.0  0.8    0 
   1  0.0    0  0.0  0.8 
   2  0.0  0.0  0.0  0.0 
------------------
  r/c   0    1    2    3 
   0  0.0 0.58 0.87    0 
   1  0.0    0 0.65 0.87 
   2  0.0  0.0  0.0 0.58 
------------------
  r/c   0    1    2    3 
   0 0.41 0.73 0.94    0 
   1  0.0    0 0.76 0.94 
   2  0.0  0.0 0.52 0.68 
------------------
  r/c   0    1    2    3 
   0 0.56 0.81 0.95    0 
   1  0.3    0 0.83 0.95 
   2  0.0 0.37 0.61 0.78 
------------------
  r/c   0    1    2    3 
   0 0.66 0.83 0.96    0 
   1 0.46    0 0.85 0.96 
   2  0.3 0.51  0.7 0.81 
------------------
  r/c   0    1    2    3 
   0  0.7 0.84 0.96    0 
   1 0.56    0 0.85 0.96 
   2 0.43  0.6 0.73 0.83 
------------------
  r/c   0    1    2    3 
   0 0.72 0.84 0.96    0 
   1  0.6    0 0.86 0.96 
   2 0.52 0.63 0.74 0.83 
------------------
  r/c   0    1    2    3 
   0 0.73 0.85 0.96    0 
   1 0.63    0 0.86 0.96 
   2 0.56 0.65 0.75

In [156]:
def eps_soft(grid: Grid, policy, eps=0.1):
    policy = [row[:] for row in policy]
    action_keys = list(grid.actions.keys())
    n_actions = len(action_keys)
    for r in range(grid.rows):
        for c in range(grid.cols):
            if policy[r][c] is None:
                continue
            ga = max(policy[r][c], key=policy[r][c].get)
            other_actions = {
                action: eps / n_actions
                for action in action_keys
                if action != ga
            }
            greedy_action = {
                ga: 1.0 - eps + eps / n_actions,
            }
            policy[r][c] = {**greedy_action, **other_actions}

    return policy

In [157]:
def policy_iteration(
    grid: Grid,
    policy=None,
    max_iters=1000,
    pass_incumbent_policy=False,
    gamma=0.9,
    theta=1e-6,
    policy_evaluation_verbose=False,
    policy_evaluation_max_iters=1000,
    eps = None,
):
    print('---------- policy_iteration ------------')
    if policy is None:
        policy = [[{'up': 1.0} for _ in range(grid.cols)] for _ in range(grid.rows)]
    i = 0
    policy_evaluation_converged = True
    changed = None
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    while changed != 0 and i < max_iters:
        V, policy_evaluation_converged = policy_evaluation(
            grid,
            policy if eps is None else eps_soft(grid, policy, eps),
            gamma,
            theta,
            verbose=policy_evaluation_verbose,
            max_iters=policy_evaluation_max_iters,
        )
        if not policy_evaluation_converged:
            break
        show_V(grid, V)
        new_policy = read_policy(
            grid, V, gamma, incumbent_policy=policy if pass_incumbent_policy else None
        )
        changed = sum(
            new_policy[r][c] != None
            and (
                max(new_policy[r][c], key=new_policy[r][c].get)
                != max(policy[r][c], key=policy[r][c].get)
            )
            for r in range(grid.rows)
            for c in range(grid.cols)
        )
        print(f'actions changed = {changed}')
        render_policy(grid, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if not policy_evaluation_converged:
        print(
            f"policy_iteration did not converge because policy_evaluation did not converge"
        )
    elif converged:
        print(f'policy_iteration converged in {i} iterations')
    else:
        print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [158]:
policy_iteration(default_grid);

---------- policy_iteration ------------
policy_evaluation converged in 82 iterations
  r/c   0    1    2    3 
   0 0.07 0.15 0.41    0 
   1 0.06    0 0.41 0.92 
   2 0.06 0.14 0.38 0.77 
------------------
actions changed = 7
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  →  ↑ 
 2  →  →  →  ↑ 
------------------
policy_evaluation converged in 22 iterations
  r/c   0    1    2    3 
   0 0.73 0.85 0.96    0 
   1 0.64    0 0.85 0.96 
   2 0.58 0.65 0.74 0.84 
------------------
actions changed = 1
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  →  ↑ 
------------------
policy_evaluation converged in 20 iterations
  r/c   0    1    2    3 
   0 0.73 0.85 0.96    0 
   1 0.64    0 0.86 0.96 
   2 0.58 0.66 0.75 0.84 
------------------
actions changed = 1
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
policy_evaluation converged in 20 iterations
  r/c   0    1    2    3 
   0 0.73 0.85 0.96    0 
   1 0.64    0 0.86 0.96 
   2 0.59 0.66 0.75 0.8